In [1]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from uuid import uuid4

import pandas as pd
import vertexai
from google import genai
from google.api_core.exceptions import Conflict, NotFound
from google.cloud import bigquery
from google.cloud import storage
from vertexai.language_models import TextEmbeddingModel

In [2]:
PROJECT_ID = "leafy-guide-497515-m4"
LOCATION = "us-central1"

BUCKET_NAME = "leafy-guide-497515-m4-vector-assets"

DATASET_ID = "product_feedback_intelligence_rag"
DOCUMENT_TABLE_ID = "product_feedback_documents"
CHUNK_TABLE_ID = "product_feedback_chunks"
INSIGHT_TABLE_ID = "product_feedback_insights"

PLANNING_MODEL = "gemini-2.5-pro"
TEXT_MODEL = "gemini-2.5-flash"
TEXT_EMBEDDING_MODEL = "text-embedding-005"

CHUNK_SIZE_CHARS = 1000
CHUNK_OVERLAP_CHARS = 180
MAX_EMBEDDING_TEXT_CHARS = 3000

GCS_SUMMARY_PREFIX = "product-feedback-intelligence-rag/summaries"

print("Configuration loaded.")
print("Project:", PROJECT_ID)
print("Location:", LOCATION)
print("Bucket:", BUCKET_NAME)
print("Planning model:", PLANNING_MODEL)
print("Text model:", TEXT_MODEL)
print("Embedding model:", TEXT_EMBEDDING_MODEL)

Configuration loaded.
Project: leafy-guide-497515-m4
Location: us-central1
Bucket: leafy-guide-497515-m4-vector-assets
Planning model: gemini-2.5-pro
Text model: gemini-2.5-flash
Embedding model: text-embedding-005


In [3]:
vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
)

google_vertex_client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
)

storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(BUCKET_NAME)

bigquery_client = bigquery.Client(project=PROJECT_ID)

text_embedding_model = TextEmbeddingModel.from_pretrained(
    TEXT_EMBEDDING_MODEL
)

print("Clients created.")
print("Bucket exists:", bucket.exists())
print("BigQuery project:", bigquery_client.project)
print("Text embedding model loaded.")

/home/lipov/projects/Python_practice_sessions_05/.venv/lib/python3.13/site-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


Clients created.
Bucket exists: True
BigQuery project: leafy-guide-497515-m4
Text embedding model loaded.


In [4]:
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = "EU"

try:
    dataset = bigquery_client.create_dataset(dataset_ref)
    print("Created dataset:", dataset.full_dataset_id)
except Conflict:
    dataset = bigquery_client.get_dataset(dataset_ref)
    print("Dataset already exists:", dataset.full_dataset_id)

document_table_ref = f"{PROJECT_ID}.{DATASET_ID}.{DOCUMENT_TABLE_ID}"
chunk_table_ref = f"{PROJECT_ID}.{DATASET_ID}.{CHUNK_TABLE_ID}"
insight_table_ref = f"{PROJECT_ID}.{DATASET_ID}.{INSIGHT_TABLE_ID}"

print("Document table:", document_table_ref)
print("Chunk table:", chunk_table_ref)
print("Insight table:", insight_table_ref)

Created dataset: leafy-guide-497515-m4:product_feedback_intelligence_rag
Document table: leafy-guide-497515-m4.product_feedback_intelligence_rag.product_feedback_documents
Chunk table: leafy-guide-497515-m4.product_feedback_intelligence_rag.product_feedback_chunks
Insight table: leafy-guide-497515-m4.product_feedback_intelligence_rag.product_feedback_insights


In [5]:
def ensure_bigquery_table(
    table_ref: str,
    schema: list[bigquery.SchemaField],
) -> bigquery.Table:
    try:
        table = bigquery_client.get_table(table_ref)
        print("Table already exists:", table.full_table_id)
        return table

    except NotFound:
        print("Table does not exist. Creating:", table_ref)

        table = bigquery.Table(
            table_ref,
            schema=schema,
        )

        created_table = bigquery_client.create_table(table)
        print("Created table:", created_table.full_table_id)

        return created_table

In [6]:
document_schema = [
    bigquery.SchemaField("document_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("document_number", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("source_type", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("product_area", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("persona", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("sentiment", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("priority", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("title", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("content", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("summary", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("created_at", "TIMESTAMP", mode="REQUIRED"),
]

chunk_schema = [
    bigquery.SchemaField("chunk_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("document_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("document_number", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("chunk_number", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("global_chunk_number", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("source_type", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("product_area", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("persona", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("sentiment", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("priority", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("title", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("chunk_text", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("chunk_char_count", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("embedding_model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("embedding", "FLOAT64", mode="REPEATED"),
    bigquery.SchemaField("created_at", "TIMESTAMP", mode="REQUIRED"),
]

insight_schema = [
    bigquery.SchemaField("insight_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("question", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("theme", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("product_area", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("severity", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("summary", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("recommendation", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("evidence_chunk_ids", "STRING", mode="REPEATED"),
    bigquery.SchemaField("evidence_document_titles", "STRING", mode="REPEATED"),
    bigquery.SchemaField("insight_json", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("created_at", "TIMESTAMP", mode="REQUIRED"),
]

document_table = ensure_bigquery_table(
    document_table_ref,
    document_schema,
)

chunk_table = ensure_bigquery_table(
    chunk_table_ref,
    chunk_schema,
)

insight_table = ensure_bigquery_table(
    insight_table_ref,
    insight_schema,
)

Table does not exist. Creating: leafy-guide-497515-m4.product_feedback_intelligence_rag.product_feedback_documents
Created table: leafy-guide-497515-m4:product_feedback_intelligence_rag.product_feedback_documents
Table does not exist. Creating: leafy-guide-497515-m4.product_feedback_intelligence_rag.product_feedback_chunks
Created table: leafy-guide-497515-m4:product_feedback_intelligence_rag.product_feedback_chunks
Table does not exist. Creating: leafy-guide-497515-m4.product_feedback_intelligence_rag.product_feedback_insights
Created table: leafy-guide-497515-m4:product_feedback_intelligence_rag.product_feedback_insights


In [7]:
def gcs_uri_from_blob_name(
    bucket_name: str,
    blob_name: str,
) -> str:
    return f"gs://{bucket_name}/{blob_name}"


def upload_text_to_gcs(
    text: str,
    *,
    blob_name: str,
    content_type: str = "application/json",
) -> str:
    blob = bucket.blob(blob_name)

    blob.upload_from_string(
        text,
        content_type=content_type,
    )

    return gcs_uri_from_blob_name(
        bucket_name=BUCKET_NAME,
        blob_name=blob_name,
    )

In [8]:
def extract_json_object(text: str) -> dict:
    cleaned = text.strip()

    if cleaned.startswith("```json"):
        cleaned = cleaned.removeprefix("```json").strip()

    if cleaned.startswith("```"):
        cleaned = cleaned.removeprefix("```").strip()

    if cleaned.endswith("```"):
        cleaned = cleaned.removesuffix("```").strip()

    start = cleaned.find("{")
    end = cleaned.rfind("}")

    if start == -1 or end == -1:
        raise ValueError(f"No JSON object found in text: {text[:500]}")

    return json.loads(cleaned[start:end + 1])

In [9]:
product_feedback_brief = """
Create a synthetic product feedback intelligence dataset for a SaaS product called FlowCart.

FlowCart helps ecommerce teams:
- manage checkout flows
- track abandoned carts
- run promotions
- synchronize inventory
- analyze conversion funnels
- use AI recommendations for product bundles

The dataset should include:
- app store reviews
- support chat transcripts
- sales call notes
- churn interview notes
- beta tester feedback
- internal product requirement notes

The goal is to build a product intelligence RAG system that helps a Product Manager:
- find repeated user pain points
- understand feature requests
- connect complaints to product areas
- generate product insights from evidence
- prioritize roadmap items
- draft PRD sections from real feedback chunks
"""

print(product_feedback_brief)


Create a synthetic product feedback intelligence dataset for a SaaS product called FlowCart.

FlowCart helps ecommerce teams:
- manage checkout flows
- track abandoned carts
- run promotions
- synchronize inventory
- analyze conversion funnels
- use AI recommendations for product bundles

The dataset should include:
- app store reviews
- support chat transcripts
- sales call notes
- churn interview notes
- beta tester feedback
- internal product requirement notes

The goal is to build a product intelligence RAG system that helps a Product Manager:
- find repeated user pain points
- understand feature requests
- connect complaints to product areas
- generate product insights from evidence
- prioritize roadmap items
- draft PRD sections from real feedback chunks



In [10]:
feedback_specs = [
    {
        "document_number": 1,
        "source_type": "app_store_reviews",
        "product_area": "checkout",
        "persona": "mobile shopper",
        "sentiment": "negative",
        "priority": "high",
        "title": "Mobile checkout reviews mentioning payment friction",
        "focus": "failed payment attempts, confusing promo code field, unclear shipping cost, abandoned checkout",
    },
    {
        "document_number": 2,
        "source_type": "support_chat_transcripts",
        "product_area": "returns",
        "persona": "support agent and merchant admin",
        "sentiment": "mixed",
        "priority": "medium",
        "title": "Support chats about returns workflow confusion",
        "focus": "return labels, manual approval steps, unclear customer notifications, repeated support questions",
    },
    {
        "document_number": 3,
        "source_type": "sales_call_notes",
        "product_area": "analytics_dashboard",
        "persona": "B2B ecommerce operations manager",
        "sentiment": "mixed",
        "priority": "high",
        "title": "Sales calls requesting better conversion analytics",
        "focus": "funnel breakdown, cohort filters, abandoned cart analysis, executive reporting, export needs",
    },
    {
        "document_number": 4,
        "source_type": "churn_interview",
        "product_area": "inventory_sync",
        "persona": "small store owner",
        "sentiment": "negative",
        "priority": "critical",
        "title": "Churn interview about inventory sync reliability",
        "focus": "delayed stock updates, overselling, marketplace mismatch, loss of trust, manual reconciliation",
    },
    {
        "document_number": 5,
        "source_type": "beta_feedback",
        "product_area": "ai_recommendations",
        "persona": "power user merchant",
        "sentiment": "positive",
        "priority": "medium",
        "title": "Beta feedback about AI bundle recommendations",
        "focus": "useful bundle ideas, missing controls, need explanations, seasonal merchandising, confidence score",
    },
    {
        "document_number": 6,
        "source_type": "internal_prd_notes",
        "product_area": "checkout",
        "persona": "product team",
        "sentiment": "neutral",
        "priority": "high",
        "title": "Internal PRD notes for checkout redesign",
        "focus": "reduce friction, simplify promo code behavior, improve payment error states, address mobile layout problems",
    },
]

pd.DataFrame(feedback_specs)

,document_number,source_type,product_area,persona,sentiment,priority,title,focus
0,1,app_store_reviews,checkout,mobile shopper,negative,high,Mobile checkout reviews mentioning payment fri...,"failed payment attempts, confusing promo code ..."
1,2,support_chat_transcripts,returns,support agent and merchant admin,mixed,medium,Support chats about returns workflow confusion,"return labels, manual approval steps, unclear ..."
2,3,sales_call_notes,analytics_dashboard,B2B ecommerce operations manager,mixed,high,Sales calls requesting better conversion analy...,"funnel breakdown, cohort filters, abandoned ca..."
3,4,churn_interview,inventory_sync,small store owner,negative,critical,Churn interview about inventory sync reliability,"delayed stock updates, overselling, marketplac..."
4,5,beta_feedback,ai_recommendations,power user merchant,positive,medium,Beta feedback about AI bundle recommendations,"useful bundle ideas, missing controls, need ex..."
5,6,internal_prd_notes,checkout,product team,neutral,high,Internal PRD notes for checkout redesign,"reduce friction, simplify promo code behavior,..."


In [11]:
def generate_feedback_document(spec: dict) -> str:
    prompt = f"""
You are a senior Product Operations analyst creating synthetic but realistic product feedback.

Product feedback brief:
{product_feedback_brief}

Document metadata:
{json.dumps(spec, indent=2, ensure_ascii=False)}

Create a realistic internal document in English.

Requirements:
- Make it synthetic but realistic.
- Do not mention that it is AI-generated.
- Include concrete user quotes, recurring pain points, product details, and operational context.
- Include enough text to be split into multiple semantic chunks.
- Use clear sections.
- Do not use markdown tables.

For the selected source type, write in the correct style:
- app_store_reviews: multiple short reviews with ratings and user language
- support_chat_transcripts: realistic chat snippets and support summaries
- sales_call_notes: account notes, objections, feature requests, buyer goals
- churn_interview: interview summary with reasons for cancellation
- beta_feedback: structured tester feedback
- internal_prd_notes: product requirement notes grounded in customer evidence
"""

    response = google_vertex_client.models.generate_content(
        model=PLANNING_MODEL,
        contents=prompt,
    )

    return response.text.strip()


feedback_documents = []

for spec in feedback_specs:
    print("=" * 100)
    print("Generating:", spec["title"])

    content = generate_feedback_document(spec)

    document = {
        "document_id": str(uuid4()),
        "document_number": spec["document_number"],
        "source_type": spec["source_type"],
        "product_area": spec["product_area"],
        "persona": spec["persona"],
        "sentiment": spec["sentiment"],
        "priority": spec["priority"],
        "title": spec["title"],
        "content": content,
        "summary": content[:700],
        "created_at": datetime.now(timezone.utc).isoformat(),
    }

    feedback_documents.append(document)

    print("Characters:", len(content))
    print(content[:500])

Generating: Mobile checkout reviews mentioning payment friction
Characters: 3825
**Document Number:** 1
**Source Type:** app_store_reviews
**Product Area:** checkout
**Persona:** mobile shopper
**Sentiment:** negative
**Priority:** high
**Title:** Mobile checkout reviews mentioning payment friction
**Focus:** failed payment attempts, confusing promo code field, unclear shipping cost, abandoned checkout

***

### Mobile Checkout Experience: Negative Review Compilation

**Date Compiled:** October 26, 2023
**Compiled By:** Product Operations
**Source:** Public App Stores (iOS &
Generating: Support chats about returns workflow confusion
Characters: 7491
**Document Number:** 2
**Source:** Support Team Synthesis
**Date:** October 26, 2023
**Author:** Maria Valerio, Support Lead
**Title:** Support chats about returns workflow confusion

### 1. Executive Summary

This document aggregates recurring themes from support chats over the past quarter (Q3 2023) related to the Returns Management modul

In [12]:
print("Documents:", len(feedback_documents))

documents_df = pd.DataFrame(
    [
        {
            "document_number": document["document_number"],
            "source_type": document["source_type"],
            "product_area": document["product_area"],
            "persona": document["persona"],
            "sentiment": document["sentiment"],
            "priority": document["priority"],
            "title": document["title"],
            "content_length": len(document["content"]),
        }
        for document in feedback_documents
    ]
)

documents_df

Documents: 6


,document_number,source_type,product_area,persona,sentiment,priority,title,content_length
0,1,app_store_reviews,checkout,mobile shopper,negative,high,Mobile checkout reviews mentioning payment fri...,3825
1,2,support_chat_transcripts,returns,support agent and merchant admin,mixed,medium,Support chats about returns workflow confusion,7491
2,3,sales_call_notes,analytics_dashboard,B2B ecommerce operations manager,mixed,high,Sales calls requesting better conversion analy...,6032
3,4,churn_interview,inventory_sync,small store owner,negative,critical,Churn interview about inventory sync reliability,5640
4,5,beta_feedback,ai_recommendations,power user merchant,positive,medium,Beta feedback about AI bundle recommendations,5877
5,6,internal_prd_notes,checkout,product team,neutral,high,Internal PRD notes for checkout redesign,7740


In [13]:
def normalize_text(text: str) -> str:
    return " ".join(text.split())


def split_text_into_chunks(
    text: str,
    *,
    chunk_size_chars: int = CHUNK_SIZE_CHARS,
    chunk_overlap_chars: int = CHUNK_OVERLAP_CHARS,
) -> list[str]:
    clean_text = normalize_text(text)

    if len(clean_text) <= chunk_size_chars:
        return [clean_text]

    chunks = []
    start = 0

    while start < len(clean_text):
        end = min(start + chunk_size_chars, len(clean_text))

        if end < len(clean_text):
            sentence_boundary = clean_text.rfind(". ", start, end)

            if sentence_boundary > start + int(chunk_size_chars * 0.6):
                end = sentence_boundary + 1

        chunk = clean_text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(clean_text):
            break

        start = max(0, end - chunk_overlap_chars)

    return chunks

In [14]:
chunk_rows_without_embeddings = []
global_chunk_number = 1

for document in feedback_documents:
    chunks = split_text_into_chunks(document["content"])

    print("=" * 100)
    print("Document:", document["title"])
    print("Chunks:", len(chunks))

    for chunk_index, chunk_text in enumerate(chunks, start=1):
        chunk_rows_without_embeddings.append(
            {
                "chunk_id": str(uuid4()),
                "document_id": document["document_id"],
                "document_number": document["document_number"],
                "chunk_number": chunk_index,
                "global_chunk_number": global_chunk_number,
                "source_type": document["source_type"],
                "product_area": document["product_area"],
                "persona": document["persona"],
                "sentiment": document["sentiment"],
                "priority": document["priority"],
                "title": document["title"],
                "chunk_text": chunk_text,
                "chunk_char_count": len(chunk_text),
                "embedding_model": TEXT_EMBEDDING_MODEL,
                "created_at": datetime.now(timezone.utc).isoformat(),
            }
        )

        global_chunk_number += 1

print("Total chunks:", len(chunk_rows_without_embeddings))

chunks_preview_df = pd.DataFrame(
    [
        {
            "global_chunk_number": row["global_chunk_number"],
            "product_area": row["product_area"],
            "source_type": row["source_type"],
            "priority": row["priority"],
            "chunk_char_count": row["chunk_char_count"],
            "preview": row["chunk_text"][:180],
        }
        for row in chunk_rows_without_embeddings
    ]
)

chunks_preview_df.head(10)

Document: Mobile checkout reviews mentioning payment friction
Chunks: 5
Document: Support chats about returns workflow confusion
Chunks: 10
Document: Sales calls requesting better conversion analytics
Chunks: 8
Document: Churn interview about inventory sync reliability
Chunks: 7
Document: Beta feedback about AI bundle recommendations
Chunks: 8
Document: Internal PRD notes for checkout redesign
Chunks: 11
Total chunks: 49


,global_chunk_number,product_area,source_type,priority,chunk_char_count,preview
0,1,checkout,app_store_reviews,high,981,**Document Number:** 1 **Source Type:** app_st...
1,2,checkout,app_store_reviews,high,883,"the application of promotional codes, and unex..."
2,3,checkout,app_store_reviews,high,911,"are fine, I used one right after on Amazon. Th..."
3,4,checkout,app_store_reviews,high,998,:** ★☆☆☆☆ **User:** runnermom_4 **Title:** Dec...
4,5,checkout,app_store_reviews,high,754,"up for a split second, and then disappears. No..."
5,6,returns,support_chat_transcripts,medium,835,**Document Number:** 2 **Source:** Support Tea...
6,7,returns,support_chat_transcripts,medium,835,"nger resolution times for our team and, more i..."
7,8,returns,support_chat_transcripts,medium,797,"r sent, and merchants initiating support chats..."
8,9,returns,support_chat_transcripts,medium,993,or issuing store credit instead of a cash refu...
9,10,returns,support_chat_transcripts,medium,927,urn 2 days ago (Order #94331). They just email...


In [15]:
def shorten_text_for_embedding(
    text: str,
    *,
    max_chars: int = MAX_EMBEDDING_TEXT_CHARS,
) -> str:
    clean_text = normalize_text(text)

    if len(clean_text) <= max_chars:
        return clean_text

    return clean_text[:max_chars].rsplit(" ", 1)[0]


def get_text_embedding(text: str) -> list[float]:
    safe_text = shorten_text_for_embedding(text)

    embeddings = text_embedding_model.get_embeddings(
        [safe_text]
    )

    return list(embeddings[0].values)

In [16]:
chunk_rows = []

for row in chunk_rows_without_embeddings:
    print(
        "Embedding chunk:",
        row["global_chunk_number"],
        "|",
        row["product_area"],
        "|",
        row["source_type"],
    )

    embedding = get_text_embedding(row["chunk_text"])

    chunk_rows.append(
        {
            **row,
            "embedding": embedding,
        }
    )

    print("Embedding dim:", len(embedding))
    print("Chunk chars:", row["chunk_char_count"])
    print("-" * 100)

print("Chunk rows with embeddings:", len(chunk_rows))

Embedding chunk: 1 | checkout | app_store_reviews
Embedding dim: 768
Chunk chars: 981
----------------------------------------------------------------------------------------------------
Embedding chunk: 2 | checkout | app_store_reviews
Embedding dim: 768
Chunk chars: 883
----------------------------------------------------------------------------------------------------
Embedding chunk: 3 | checkout | app_store_reviews
Embedding dim: 768
Chunk chars: 911
----------------------------------------------------------------------------------------------------
Embedding chunk: 4 | checkout | app_store_reviews
Embedding dim: 768
Chunk chars: 998
----------------------------------------------------------------------------------------------------
Embedding chunk: 5 | checkout | app_store_reviews
Embedding dim: 768
Chunk chars: 754
----------------------------------------------------------------------------------------------------
Embedding chunk: 6 | returns | support_chat_transcripts
Embedding

In [17]:
document_rows = []

for document in feedback_documents:
    document_rows.append(
        {
            "document_id": document["document_id"],
            "document_number": document["document_number"],
            "source_type": document["source_type"],
            "product_area": document["product_area"],
            "persona": document["persona"],
            "sentiment": document["sentiment"],
            "priority": document["priority"],
            "title": document["title"],
            "content": document["content"],
            "summary": document["summary"],
            "created_at": document["created_at"],
        }
    )

bigquery_client.query(
    f"TRUNCATE TABLE `{document_table_ref}`",
    location=dataset.location,
).result()

bigquery_client.query(
    f"TRUNCATE TABLE `{chunk_table_ref}`",
    location=dataset.location,
).result()

document_errors = bigquery_client.insert_rows_json(
    document_table_ref,
    document_rows,
)

chunk_errors = bigquery_client.insert_rows_json(
    chunk_table_ref,
    chunk_rows,
)

if document_errors:
    print("Document insert errors:")
    print(document_errors)
else:
    print("Inserted document rows:", len(document_rows))

if chunk_errors:
    print("Chunk insert errors:")
    print(chunk_errors)
else:
    print("Inserted chunk rows:", len(chunk_rows))

Inserted document rows: 6
Inserted chunk rows: 49


In [18]:
sql = f"""
SELECT
  (SELECT COUNT(*) FROM `{document_table_ref}`) AS document_count,
  (SELECT COUNT(*) FROM `{chunk_table_ref}`) AS chunk_count,
  (
    SELECT ARRAY_LENGTH(embedding)
    FROM `{chunk_table_ref}`
    LIMIT 1
  ) AS embedding_length
"""

result = list(
    bigquery_client.query(
        sql,
        location=dataset.location,
    )
)[0]

print("Document count:", result.document_count)
print("Chunk count:", result.chunk_count)
print("Embedding length:", result.embedding_length)

Document count: 6
Chunk count: 49
Embedding length: 768


In [19]:
def search_feedback_chunks(
    query: str,
    *,
    top_k: int = 8,
    product_area: str | None = None,
    source_type: str | None = None,
    priority: str | None = None,
) -> list[dict]:
    query_embedding = get_text_embedding(query)

    filters = []
    query_parameters = [
        bigquery.ArrayQueryParameter(
            "query_embedding",
            "FLOAT64",
            query_embedding,
        ),
        bigquery.ScalarQueryParameter(
            "top_k",
            "INT64",
            top_k,
        ),
    ]

    if product_area is not None:
        filters.append("product_area = @product_area")
        query_parameters.append(
            bigquery.ScalarQueryParameter(
                "product_area",
                "STRING",
                product_area,
            )
        )

    if source_type is not None:
        filters.append("source_type = @source_type")
        query_parameters.append(
            bigquery.ScalarQueryParameter(
                "source_type",
                "STRING",
                source_type,
            )
        )

    if priority is not None:
        filters.append("priority = @priority")
        query_parameters.append(
            bigquery.ScalarQueryParameter(
                "priority",
                "STRING",
                priority,
            )
        )

    where_sql = ""

    if filters:
        where_sql = "WHERE " + " AND ".join(filters)

    sql = f"""
    SELECT
      base.chunk_id,
      base.document_id,
      base.document_number,
      base.chunk_number,
      base.global_chunk_number,
      base.source_type,
      base.product_area,
      base.persona,
      base.sentiment,
      base.priority,
      base.title,
      base.chunk_text,
      distance
    FROM VECTOR_SEARCH(
      (
        SELECT *
        FROM `{chunk_table_ref}`
        {where_sql}
      ),
      'embedding',
      (SELECT @query_embedding AS embedding),
      top_k => @top_k,
      distance_type => 'COSINE'
    )
    ORDER BY distance ASC
    """

    job_config = bigquery.QueryJobConfig(
        query_parameters=query_parameters,
    )

    query_job = bigquery_client.query(
        sql,
        job_config=job_config,
        location=dataset.location,
    )

    return [
        {
            "chunk_id": row.chunk_id,
            "document_id": row.document_id,
            "document_number": row.document_number,
            "chunk_number": row.chunk_number,
            "global_chunk_number": row.global_chunk_number,
            "source_type": row.source_type,
            "product_area": row.product_area,
            "persona": row.persona,
            "sentiment": row.sentiment,
            "priority": row.priority,
            "title": row.title,
            "chunk_text": row.chunk_text,
            "distance": row.distance,
        }
        for row in query_job
    ]

In [20]:
query = """
Users are complaining that checkout is confusing on mobile,
promo codes behave unexpectedly, payment errors are unclear,
and shipping costs appear too late.
"""

search_results = search_feedback_chunks(
    query,
    top_k=8,
)

print("Query:")
print(query)

print("\nSearch results:", len(search_results))

for result in search_results:
    print("=" * 100)
    print("Distance:", round(result["distance"], 4))
    print("Product area:", result["product_area"])
    print("Source:", result["source_type"])
    print("Priority:", result["priority"])
    print("Document:", result["title"])
    print("Global chunk:", result["global_chunk_number"])
    print(result["chunk_text"][:900])

Query:

Users are complaining that checkout is confusing on mobile,
promo codes behave unexpectedly, payment errors are unclear,
and shipping costs appear too late.


Search results: 8
Distance: 0.1549
Product area: checkout
Source: app_store_reviews
Priority: high
Document: Mobile checkout reviews mentioning payment friction
Global chunk: 1
**Document Number:** 1 **Source Type:** app_store_reviews **Product Area:** checkout **Persona:** mobile shopper **Sentiment:** negative **Priority:** high **Title:** Mobile checkout reviews mentioning payment friction **Focus:** failed payment attempts, confusing promo code field, unclear shipping cost, abandoned checkout *** ### Mobile Checkout Experience: Negative Review Compilation **Date Compiled:** October 26, 2023 **Compiled By:** Product Operations **Source:** Public App Stores (iOS & Android) **Period:** Q3 2023 **Analyst Summary:** This document collates recent one and two-star app store reviews specifically concerning the mobile checkout

In [21]:
filtered_results = search_feedback_chunks(
    query,
    top_k=5,
    product_area="checkout",
    priority="high",
)

print("Filtered results:", len(filtered_results))

for result in filtered_results:
    print("=" * 100)
    print("Distance:", round(result["distance"], 4))
    print("Product area:", result["product_area"])
    print("Source:", result["source_type"])
    print("Priority:", result["priority"])
    print("Document:", result["title"])
    print(result["chunk_text"][:700])

Filtered results: 5
Distance: 0.1549
Product area: checkout
Source: app_store_reviews
Priority: high
Document: Mobile checkout reviews mentioning payment friction
**Document Number:** 1 **Source Type:** app_store_reviews **Product Area:** checkout **Persona:** mobile shopper **Sentiment:** negative **Priority:** high **Title:** Mobile checkout reviews mentioning payment friction **Focus:** failed payment attempts, confusing promo code field, unclear shipping cost, abandoned checkout *** ### Mobile Checkout Experience: Negative Review Compilation **Date Compiled:** October 26, 2023 **Compiled By:** Product Operations **Source:** Public App Stores (iOS & Android) **Period:** Q3 2023 **Analyst Summary:** This document collates recent one and two-star app store reviews specifically concerning the mobile checkout process. A clear and alarming pattern of us
Distance: 0.1809
Product area: checkout
Source: app_store_reviews
Priority: high
Document: Mobile checkout reviews mentioning payment fr

In [22]:
def build_rag_context_from_chunks(results: list[dict]) -> str:
    parts = []

    for index, result in enumerate(results, start=1):
        parts.append(
            "\n".join(
                [
                    f"[SOURCE {index}]",
                    f"Document title: {result['title']}",
                    f"Source type: {result['source_type']}",
                    f"Product area: {result['product_area']}",
                    f"Persona: {result['persona']}",
                    f"Sentiment: {result['sentiment']}",
                    f"Priority: {result['priority']}",
                    f"Global chunk number: {result['global_chunk_number']}",
                    f"Chunk number in document: {result['chunk_number']}",
                    f"Distance: {result['distance']:.4f}",
                    f"Chunk text: {result['chunk_text']}",
                ]
            )
        )

    return "\n\n---\n\n".join(parts)

In [23]:
def answer_product_question(
    product_question: str,
    *,
    top_k: int = 8,
    product_area: str | None = None,
    source_type: str | None = None,
    priority: str | None = None,
) -> dict:
    results = search_feedback_chunks(
        product_question,
        top_k=top_k,
        product_area=product_area,
        source_type=source_type,
        priority=priority,
    )

    context = build_rag_context_from_chunks(results)

    prompt = f"""
You are a senior Product Manager assistant.

Answer the product question using only the retrieved feedback chunks.

Product question:
{product_question}

Retrieved feedback chunks:
{context}

Return a structured product insight with:
1. Short answer
2. Main customer pain points
3. Evidence from sources
4. Product areas affected
5. Severity assessment
6. Recommended roadmap action
7. Suggested PRD bullet points
8. Sources used, referencing SOURCE numbers

Rules:
- Stay grounded in the retrieved chunks.
- Do not invent customer evidence outside the chunks.
- If evidence is weak, say what additional feedback is needed.
"""

    response = google_vertex_client.models.generate_content(
        model=TEXT_MODEL,
        contents=prompt,
    )

    return {
        "product_question": product_question,
        "answer": response.text,
        "search_results": results,
        "rag_context": context,
    }

In [24]:
product_question = """
Should we prioritize a checkout redesign next sprint?
Use customer feedback to explain the main pain points, severity, and what the PRD should include.
"""

product_rag_result = answer_product_question(
    product_question,
    top_k=8,
    product_area="checkout",
)

print("PRODUCT QUESTION:")
print(product_rag_result["product_question"])

print("\nPRODUCT RAG ANSWER:")
print(product_rag_result["answer"])

PRODUCT QUESTION:

Should we prioritize a checkout redesign next sprint?
Use customer feedback to explain the main pain points, severity, and what the PRD should include.


PRODUCT RAG ANSWER:
Yes, based on the customer feedback, prioritizing a checkout redesign next sprint is highly recommended.

**1. Short answer**
A checkout redesign should be prioritized next sprint due to significant user friction, a 12% drop-off rate, and direct revenue loss attributed to the current clunky, confusing, and often broken experience, particularly on mobile devices. Addressing these issues will directly impact conversion rates and reduce operational burden.

**2. Main customer pain points**
*   **General Checkout Friction & Cognitive Load:** The process is multi-step, too long, requires too many clicks, forces users to re-enter saved information, and has an unintuitive layout (SOURCE 1).
*   **Poor Mobile Experience:** Elements are misaligned, buttons are hard to tap, "Continue to Payment" buttons ca

In [25]:
def generate_product_insight_json(rag_result: dict) -> dict:
    evidence_chunk_ids = [
        result["chunk_id"]
        for result in rag_result["search_results"]
    ]

    evidence_document_titles = sorted(
        {
            result["title"]
            for result in rag_result["search_results"]
        }
    )

    prompt = f"""
You are a product strategy analyst.

Use the product question, retrieved context, and RAG answer below.

Product question:
{rag_result["product_question"]}

Retrieved context:
{rag_result["rag_context"]}

RAG answer:
{rag_result["answer"]}

Return only valid JSON:

{{
  "theme": "short theme name",
  "product_area": "product area",
  "severity": "low | medium | high | critical",
  "summary": "short insight summary",
  "recommendation": "specific roadmap recommendation",
  "evidence_summary": ["evidence point 1", "evidence point 2"],
  "prd_bullets": ["prd bullet 1", "prd bullet 2"],
  "open_questions": ["question 1", "question 2"]
}}

Rules:
- Stay grounded in the retrieved chunks.
- Do not use markdown.
"""

    response = google_vertex_client.models.generate_content(
        model=PLANNING_MODEL,
        contents=prompt,
    )

    insight_data = extract_json_object(response.text)

    return {
        "insight_id": str(uuid4()),
        "question": rag_result["product_question"],
        "theme": insight_data.get("theme"),
        "product_area": insight_data.get("product_area"),
        "severity": insight_data.get("severity"),
        "summary": insight_data.get("summary"),
        "recommendation": insight_data.get("recommendation"),
        "evidence_chunk_ids": evidence_chunk_ids,
        "evidence_document_titles": evidence_document_titles,
        "insight_json": json.dumps(
            insight_data,
            ensure_ascii=False,
            default=str,
        ),
        "created_at": datetime.now(timezone.utc).isoformat(),
    }


product_insight = generate_product_insight_json(product_rag_result)

print(json.dumps(product_insight, indent=2, ensure_ascii=False))

{
  "insight_id": "db777a25-4a84-4057-a72e-38243bfac6a7",
  "question": "\nShould we prioritize a checkout redesign next sprint?\nUse customer feedback to explain the main pain points, severity, and what the PRD should include.\n",
  "theme": "Checkout Friction & Revenue Loss",
  "product_area": "Checkout",
  "severity": "high",
  "summary": "The current checkout flow is causing a 12% conversion drop-off, direct revenue loss, and high support burden due to a clunky, multi-step process with a poor mobile experience, confusing promo code UX, and unhelpful payment error messages.",
  "recommendation": "Prioritize a checkout redesign in the next sprint, focusing on a streamlined, mobile-first experience with clearer user feedback to directly improve conversion rates and reduce operational costs.",
  "evidence_chunk_ids": [
    "9a134599-3cc6-48fe-96b3-28e57716e4ea",
    "62ab9685-d8a2-4e39-9d4a-2db88a1bf0b0",
    "2d751b68-3aa0-4e4e-9bb5-40879af980d2",
    "1fce0c71-0dc7-40d4-a644-67fb22d7

In [26]:
bigquery_client.query(
    f"TRUNCATE TABLE `{insight_table_ref}`",
    location=dataset.location,
).result()

insight_errors = bigquery_client.insert_rows_json(
    insight_table_ref,
    [product_insight],
)

if insight_errors:
    print("Insight insert errors:")
    print(insight_errors)
else:
    print("Inserted insight rows: 1")

Inserted insight rows: 1


In [27]:
sql = f"""
SELECT
  product_area,
  source_type,
  sentiment,
  priority,
  COUNT(*) AS chunk_count,
  ROUND(AVG(chunk_char_count), 2) AS avg_chunk_chars
FROM `{chunk_table_ref}`
GROUP BY
  product_area,
  source_type,
  sentiment,
  priority
ORDER BY
  product_area,
  priority,
  chunk_count DESC
"""

feedback_analytics_df = bigquery_client.query(
    sql,
    location=dataset.location,
).to_dataframe()

feedback_analytics_df

/home/lipov/projects/Python_practice_sessions_05/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,product_area,source_type,sentiment,priority,chunk_count,avg_chunk_chars
0,ai_recommendations,beta_feedback,positive,medium,8,889.00
1,analytics_dashboard,sales_call_notes,mixed,high,8,905.88
2,checkout,internal_prd_notes,neutral,high,11,859.82
3,checkout,app_store_reviews,negative,high,5,905.40
4,inventory_sync,churn_interview,negative,critical,7,954.29
5,returns,support_chat_transcripts,mixed,medium,10,906.50


In [28]:
sql = f"""
SELECT
  product_area,
  COUNT(*) AS chunk_count,
  COUNT(DISTINCT document_id) AS document_count,
  COUNTIF(sentiment = 'negative') AS negative_chunks,
  COUNTIF(priority IN ('high', 'critical')) AS high_priority_chunks
FROM `{chunk_table_ref}`
GROUP BY product_area
ORDER BY
  high_priority_chunks DESC,
  negative_chunks DESC,
  chunk_count DESC
"""

product_area_summary_df = bigquery_client.query(
    sql,
    location=dataset.location,
).to_dataframe()

product_area_summary_df

/home/lipov/projects/Python_practice_sessions_05/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,product_area,chunk_count,document_count,negative_chunks,high_priority_chunks
0,checkout,16,2,5,16
1,analytics_dashboard,8,1,0,8
2,inventory_sync,7,1,7,7
3,returns,10,1,0,0
4,ai_recommendations,8,1,0,0


In [29]:
test_questions = [
    "What do customers say about inventory sync reliability and overselling?",
    "Which analytics dashboard improvements are requested by B2B buyers?",
    "What controls do beta users want for AI bundle recommendations?",
    "What repeated support problems happen in the returns workflow?",
]

for question in test_questions:
    print("\n" + "=" * 100)
    print("QUESTION:")
    print(question)

    results = search_feedback_chunks(
        question,
        top_k=4,
    )

    for result in results:
        print(
            "distance:",
            round(result["distance"], 4),
            "| area:",
            result["product_area"],
            "| source:",
            result["source_type"],
            "| priority:",
            result["priority"],
            "| chunk:",
            result["global_chunk_number"],
        )


QUESTION:
What do customers say about inventory sync reliability and overselling?
distance: 0.3082 | area: inventory_sync | source: churn_interview | priority: critical | chunk: 24
distance: 0.3115 | area: inventory_sync | source: churn_interview | priority: critical | chunk: 27
distance: 0.32 | area: inventory_sync | source: churn_interview | priority: critical | chunk: 26
distance: 0.3212 | area: inventory_sync | source: churn_interview | priority: critical | chunk: 30

QUESTION:
Which analytics dashboard improvements are requested by B2B buyers?
distance: 0.2513 | area: analytics_dashboard | source: sales_call_notes | priority: high | chunk: 16
distance: 0.3072 | area: analytics_dashboard | source: sales_call_notes | priority: high | chunk: 17
distance: 0.3099 | area: analytics_dashboard | source: sales_call_notes | priority: high | chunk: 22
distance: 0.3686 | area: analytics_dashboard | source: sales_call_notes | priority: high | chunk: 23

QUESTION:
What controls do beta users w

In [30]:
def generate_roadmap_memo() -> str:
    summary_context = {
        "product_area_summary": product_area_summary_df.to_dict(orient="records"),
        "feedback_analytics": feedback_analytics_df.to_dict(orient="records"),
        "saved_product_insight": product_insight,
    }

    prompt = f"""
You are a Head of Product.

Create a concise roadmap memo based on this product feedback intelligence context:

{json.dumps(summary_context, indent=2, ensure_ascii=False, default=str)}

Write:
1. Executive summary
2. Top product risks
3. Recommended next sprint priority
4. Product areas to investigate next
5. Evidence limitations
6. Suggested follow-up research questions

Be practical and grounded in the data.
"""

    response = google_vertex_client.models.generate_content(
        model=PLANNING_MODEL,
        contents=prompt,
    )

    return response.text.strip()


roadmap_memo = generate_roadmap_memo()

print(roadmap_memo)

**TO:** Product Leadership & Engineering Leads
**FROM:** Head of Product
**DATE:** July 1, 2026
**SUBJECT:** Roadmap Memo: Q3 Priorities Based on Product Intelligence

### 1. Executive Summary

Our latest product intelligence analysis reveals two critical, high-impact areas requiring immediate attention: **Inventory Sync** and **Checkout**. Feedback clearly shows that failures in Inventory Sync are a direct cause of customer churn, while significant friction in our Checkout flow is causing a 12% conversion drop-off and direct revenue loss.

Based on the detailed pre-work available, we will prioritize a **Checkout Redesign** for the next development sprint. Concurrently, we must launch an urgent investigation into the root causes of our **Inventory Sync** failures to tee it up as our next major initiative.

### 2. Top Product Risks

The data highlights two clear and present dangers to the business:

*   **Customer Churn from Inventory Sync:** We have "critical" priority feedback from ch

In [31]:
notebook_summary = {
    "project_id": PROJECT_ID,
    "location": LOCATION,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "notebook": "62_w_google_cloud_product_feedback_intelligence_rag.ipynb",
    "models": {
        "planning_model": PLANNING_MODEL,
        "text_model": TEXT_MODEL,
        "text_embedding_model": TEXT_EMBEDDING_MODEL,
    },
    "bigquery": {
        "dataset": DATASET_ID,
        "document_table": document_table_ref,
        "chunk_table": chunk_table_ref,
        "insight_table": insight_table_ref,
    },
    "product_feedback_brief": product_feedback_brief,
    "document_count": len(feedback_documents),
    "chunk_count": len(chunk_rows),
    "product_rag_result": product_rag_result,
    "product_insight": product_insight,
    "roadmap_memo": roadmap_memo,
    "product_area_summary": product_area_summary_df.to_dict(orient="records"),
}

summary_json = json.dumps(
    notebook_summary,
    indent=2,
    ensure_ascii=False,
    default=str,
)

timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
blob_name = f"{GCS_SUMMARY_PREFIX}/summary_{timestamp}.json"

summary_gcs_uri = upload_text_to_gcs(
    summary_json,
    blob_name=blob_name,
)

print("Notebook summary saved to:")
print(summary_gcs_uri)

Notebook summary saved to:
gs://leafy-guide-497515-m4-vector-assets/product-feedback-intelligence-rag/summaries/summary_20260701_145524.json


In [32]:
print("Product Feedback Intelligence RAG notebook completed.")
print("=" * 100)

print("Documents:", len(feedback_documents))
print("Chunks:", len(chunk_rows))

print("\nBigQuery tables:")
print("-", document_table_ref)
print("-", chunk_table_ref)
print("-", insight_table_ref)

print("\nProduct area summary:")

for row in product_area_summary_df.to_dict(orient="records"):
    print("=" * 100)
    print("Product area:", row["product_area"])
    print("Chunks:", row["chunk_count"])
    print("Documents:", row["document_count"])
    print("Negative chunks:", row["negative_chunks"])
    print("High priority chunks:", row["high_priority_chunks"])

print("\nExample Product RAG answer:")
print(product_rag_result["answer"][:1400])

print("\nRoadmap memo:")
print(roadmap_memo[:1400])

print("\nSummary GCS URI:")
print(summary_gcs_uri)

Product Feedback Intelligence RAG notebook completed.
Documents: 6
Chunks: 49

BigQuery tables:
- leafy-guide-497515-m4.product_feedback_intelligence_rag.product_feedback_documents
- leafy-guide-497515-m4.product_feedback_intelligence_rag.product_feedback_chunks
- leafy-guide-497515-m4.product_feedback_intelligence_rag.product_feedback_insights

Product area summary:
Product area: checkout
Chunks: 16
Documents: 2
Negative chunks: 5
High priority chunks: 16
Product area: analytics_dashboard
Chunks: 8
Documents: 1
Negative chunks: 0
High priority chunks: 8
Product area: inventory_sync
Chunks: 7
Documents: 1
Negative chunks: 7
High priority chunks: 7
Product area: returns
Chunks: 10
Documents: 1
Negative chunks: 0
High priority chunks: 0
Product area: ai_recommendations
Chunks: 8
Documents: 1
Negative chunks: 0
High priority chunks: 0

Example Product RAG answer:
Yes, based on the customer feedback, prioritizing a checkout redesign next sprint is highly recommended.

**1. Short answer**
A